# 🔬 Caterva2 Dataset Exploration Agent (Chat UI)

A notebook-native chat interface for exploring Blosc2/Caterva2/HDF5 datasets.

---

### Quick start
1. **Run Cell 1** once to import the `Agent` and helpers.
2. **Run Cell 2** to create or reset the agent instance.
3. **Run Cell 3** to launch a scrollable chat UI with an input box at the bottom.
4. Ask questions continuously and scroll to compare previous interactions.

> **Requires:** A `.env` file at project root with `GROQ_API_KEY=...`


In [4]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
# Adds caterva2_agent/ to Python's module search path.
# Defines ask() — the notebook equivalent of main.py's interactive loop.
import sys, os

agent_dir = os.getcwd()
if agent_dir not in sys.path:
    sys.path.insert(0, agent_dir)

from agent import Agent

MAX_INPUT_CHARS = 5000  # same guard as main.py

def ask(message: str) -> None:
    """Send a message to the agent, mirroring all logic from main.py."""
    message = message.strip()
    if not message:
        print('[No input provided]')
        return
    if message.lower() in ['quit', 'exit']:
        print('Goodbye! Conversation and token counter reset.')
        agent.reset()
        return
    # reset calls agent.reset() directly — does NOT send the word to the LLM
    if message.lower() == 'reset':
        agent.reset()
        print('🔄 Conversation reset. Start a new question below.')
        return
    if len(message) > MAX_INPUT_CHARS:
        print(f'[Input too long: {len(message)} chars. Max is {MAX_INPUT_CHARS}]')
        return
    if len(message) > 2000:
        print(f'[Warning: Long input ({len(message)} chars) — may take longer]')
    try:
        response = agent.run(message)
        print(response)
    except Exception as e:
        print(f'[Error: {type(e).__name__}: {e}]')
        print("Try again or call ask('reset') to clear the conversation.")
        agent.reset()  # auto-reset on unhandled exception, same as main.py

print('Setup complete — ask() helper ready.')


Setup complete — ask() helper ready.


In [5]:
# ── Cell 2: Create Agent ───────────────────────────────────────────────────────
# Creates a new Agent instance with an empty conversation history.
# Re-run this cell any time you want to start a completely fresh session.
agent = Agent()
print("✅ Agent ready.")

✅ Agent ready.


In [6]:
# ── Cell 3: Launch chat UI (scrollable history + bottom input) ─────────────
import html
import ipywidgets as widgets
from IPython.display import Markdown, display

# Layout strategy:
# - outer VBox has fixed viewport-relative height
# - chat panel flexes and scrolls
# - composer (input/buttons) stays pinned at bottom of this widget
ui_height = '72vh'

chat_output = widgets.Output(layout=widgets.Layout(
    border='1px solid #ddd',
    padding='10px',
    margin='0 0 8px 0',
    overflow='auto',
    flex='1 1 auto',
    min_height='220px'
))

# Optional HTML transcript mirror to guarantee strict in-box wrapping/scrolling for long text
# (kept simple so we still render markdown/plots through chat_output)

user_input = widgets.Text(
    placeholder='Ask about datasets, metadata, slices, etc...',
    description='You:',
    layout=widgets.Layout(width='100%')
)

send_button = widgets.Button(description='Send', button_style='primary')
clear_button = widgets.Button(description='Clear chat UI')
reset_button = widgets.Button(description='Reset agent memory', button_style='warning')
status = widgets.HTML(value='<i>Ready.</i>')

def append_message(role: str, text: str, as_markdown: bool = False) -> None:
    with chat_output:
        if role == 'user':
            print(f'🧑 You: {text}')
        elif role == 'agent':
            print('🤖 Agent:')
            if as_markdown:
                display(Markdown(text))
            else:
                print(text)
        else:
            print(f'⚙️  {text}')
        print('')

def handle_message(message: str) -> None:
    message = message.strip()
    if not message:
        return

    if message.lower() in {'quit', 'exit'}:
        append_message('system', 'Session closed in UI (agent memory preserved).')
        return

    if message.lower() == 'reset':
        agent.reset()
        append_message('system', 'Agent conversation memory reset.')
        return

    if len(message) > MAX_INPUT_CHARS:
        append_message('system', f'Input too long: {len(message)} chars. Max is {MAX_INPUT_CHARS}.')
        return

    append_message('user', message)
    status.value = '<i>Thinking…</i>'
    try:
        response = agent.run(message)
        append_message('agent', response, as_markdown=True)
        status.value = '<i>Ready.</i>'
    except Exception as e:
        append_message('system', f'Error: {type(e).__name__}: {e}')
        append_message('system', "Try again or type 'reset' in the input box.")
        agent.reset()
        status.value = '<i>Error handled. Agent memory reset.</i>'

def on_send(_):
    message = user_input.value
    user_input.value = ''
    handle_message(message)

def on_clear(_):
    chat_output.clear_output()
    append_message('system', 'Cleared visible chat history (agent memory still active).')

def on_reset(_):
    agent.reset()
    append_message('system', 'Agent memory reset from button.')

def on_enter(change):
    message = (change.get('new') or '').strip()
    if not message:
        return
    user_input.value = ''
    handle_message(message)

user_input.continuous_update = False
send_button.on_click(on_send)
clear_button.on_click(on_clear)
reset_button.on_click(on_reset)
user_input.observe(on_enter, names='value')

composer = widgets.VBox([
    user_input,
    widgets.HBox([send_button, clear_button, reset_button]),
    status,
], layout=widgets.Layout(flex='0 0 auto'))

ui = widgets.VBox(
    [chat_output, composer],
    layout=widgets.Layout(
        height=ui_height,
        border='1px solid #ccc',
        padding='8px',
        overflow='hidden'
    ),
)

display(ui)
append_message('system', 'Chat UI ready. Type in the input box and press Enter or Send.')

